# FAST-UAV - Fixed-Wing Design Geometry Partial check
*Author: Fernando Urquía Cutillas - 2026* <br>

This notebook checks the partials of the optimization groups through the built-in function check partials of OpenMDAO. It consists on comparing the relative error of the analytical method computed by the user in `def compute_partials()` against the finite difference computed with the data in `def compute()`

## 1. Fuselage Geometry

In [ ]:
import openmdao.api as om
import numpy as np

from fastuav.models.geometry.geometry_fixedwing import FuselageGeometry  # adjust path if needed

REL_TOL = 1e-5

# 1. Initialize
prob = om.Problem()
prob.model.add_subsystem("fus_geom", FuselageGeometry(), promotes=["*"])
prob.setup(force_alloc_complex=True)

# 2. Override the np.nan defaults
prob.set_val("data:geometry:wing:root:TE:x", 0.5, units="m")
prob.set_val("data:geometry:tail:horizontal:root:TE:x", 1.3, units="m")
# Fuselage inputs have defaults, but set them explicitly for clarity
prob.set_val("data:geometry:fuselage:diameter:k", 0.2)
prob.set_val("data:geometry:fuselage:fineness", 5.0)

# 3. Run once
prob.run_model()
print("Fuselage Geometry Results:")
print("  Length:       ", prob.get_val("data:geometry:fuselage:length"), "m")
print("  Diameter Mid: ", prob.get_val("data:geometry:fuselage:diameter:mid"), "m")
print("  Diameter Tip: ", prob.get_val("data:geometry:fuselage:diameter:tip"), "m")
print("  Length Nose:  ", prob.get_val("data:geometry:fuselage:length:nose"), "m")
print("  Length Mid:   ", prob.get_val("data:geometry:fuselage:length:mid"), "m")
print("  Length Rear:  ", prob.get_val("data:geometry:fuselage:length:rear"), "m\n")

# 4. Check partials (central diff avoids forward-FD false-flags)
data = prob.check_partials(
    compact_print=True,
    show_only_incorrect=False,
    method="fd",
    form="central",
    step=1e-6,
)

# 5. Detailed results
print("\n" + "=" * 70)
print("DETAILED RESULTS  (rel-error threshold = %.0e)" % REL_TOL)
print("=" * 70 + "\n")

n_fail = 0
for comp_name in data:
    for (out, inp), err in data[comp_name].items():
        J_fwd = np.asarray(err["J_fwd"])
        J_fd  = np.asarray(err["J_fd"])
        rel_error = err["rel error"].forward
        abs_error = err["abs error"].forward

        if J_fwd.size == 1:
            a_str, fd_str = f"{J_fwd.item():12.6e}", f"{J_fd.item():12.6e}"
        else:
            a_str  = f"vector{J_fwd.shape} max|.|={np.max(np.abs(J_fwd)):.3e}"
            fd_str = f"vector{J_fd.shape} max|.|={np.max(np.abs(J_fd)):.3e}"

        is_zero = (np.max(np.abs(J_fwd)) == 0.0) and (np.max(np.abs(J_fd)) == 0.0)
        status = "PASS" if (is_zero or rel_error < REL_TOL) else "FAIL"
        n_fail += status == "FAIL"

        print(f"d({out}) / d({inp})")
        print(f"  Analytic:  {a_str}")
        print(f"  FD:        {fd_str}")
        print(f"  Rel Error: {rel_error:12.6e}")
        print(f"  Abs Error: {abs_error:12.6e}")
        print(f"  {status}\n")

print("=" * 70)
print("ALL PASSED" if n_fail == 0 else f"{n_fail} FAILED")
print("=" * 70)